# Smart Material Planning: Backorder Prediction with Machine Learning

## Data Cleaning and Preparation

### Import libraries

In [2]:
import pandas as pd
import numpy as np # Numerical operations
import matplotlib.pyplot as plt # Data visualization
import seaborn as sns
from scipy import stats

In [4]:
pd.set_option("display.max_columns", None) # Display all columns when inspecting the dataset
pd.set_option("display.float_format", lambda x: f"{x:,.2f}") #Two decimals shown

### Load data

The project includes separate training and testing datasets.

In [5]:
train_path = "../data/raw/Training_BOP.csv"
test_path = "../data/raw/Testing_BOP.csv"

df_train = pd.read_csv(train_path, low_memory=False) #Inspects the entire file before assigning data types in both of the datasets
df_test = pd.read_csv(test_path, low_memory=False)

### Initial dataset inspection

In [6]:
print(f"Training dataset shape: {df_train.shape}")

Training dataset shape: (1687861, 23)


In [7]:
print(f"Testing dataset shape: {df_test.shape}")

Testing dataset shape: (242076, 23)


In [8]:
df_train.head()

,sku,national_inv,lead_time,in_transit_qty,forecast_3_month,forecast_6_month,forecast_9_month,sales_1_month,sales_3_month,sales_6_month,sales_9_month,min_bank,potential_issue,pieces_past_due,perf_6_month_avg,perf_12_month_avg,local_bo_qty,deck_risk,oe_constraint,ppap_risk,stop_auto_buy,rev_stop,went_on_backorder
0,1026827,0.00,NaN,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,No,0.00,-99.00,-99.00,0.00,No,No,No,Yes,No,No
1,1043384,2.00,9.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,No,0.00,0.99,0.99,0.00,No,No,No,Yes,No,No
2,1043696,2.00,NaN,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,No,0.00,-99.00,-99.00,0.00,Yes,No,No,Yes,No,No
3,1043852,7.00,8.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,No,0.00,0.10,0.13,0.00,No,No,No,Yes,No,No
4,1044048,8.00,NaN,0.00,0.00,0.00,0.00,0.00,0.00,0.00,4.00,2.00,No,0.00,-99.00,-99.00,0.00,Yes,No,No,Yes,No,No


In [9]:
df_test.head()

,sku,national_inv,lead_time,in_transit_qty,forecast_3_month,forecast_6_month,forecast_9_month,sales_1_month,sales_3_month,sales_6_month,sales_9_month,min_bank,potential_issue,pieces_past_due,perf_6_month_avg,perf_12_month_avg,local_bo_qty,deck_risk,oe_constraint,ppap_risk,stop_auto_buy,rev_stop,went_on_backorder
0,3285085,62.00,NaN,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,No,0.00,-99.00,-99.00,0.00,Yes,No,No,Yes,No,No
1,3285131,9.00,NaN,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,No,0.00,-99.00,-99.00,0.00,No,No,Yes,No,No,No
2,3285358,17.00,8.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,No,0.00,0.92,0.95,0.00,No,No,No,Yes,No,No
3,3285517,9.00,2.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,2.00,0.00,No,0.00,0.78,0.75,0.00,No,No,Yes,Yes,No,No
4,3285608,2.00,8.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,No,0.00,0.54,0.71,0.00,No,No,No,Yes,No,No


In [10]:
print("Training columns:")
print(df_train.columns.tolist())

Training columns:
['sku', 'national_inv', 'lead_time', 'in_transit_qty', 'forecast_3_month', 'forecast_6_month', 'forecast_9_month', 'sales_1_month', 'sales_3_month', 'sales_6_month', 'sales_9_month', 'min_bank', 'potential_issue', 'pieces_past_due', 'perf_6_month_avg', 'perf_12_month_avg', 'local_bo_qty', 'deck_risk', 'oe_constraint', 'ppap_risk', 'stop_auto_buy', 'rev_stop', 'went_on_backorder']


In [12]:
print("Testing columns:")
print(df_test.columns.tolist())

Testing columns:
['sku', 'national_inv', 'lead_time', 'in_transit_qty', 'forecast_3_month', 'forecast_6_month', 'forecast_9_month', 'sales_1_month', 'sales_3_month', 'sales_6_month', 'sales_9_month', 'min_bank', 'potential_issue', 'pieces_past_due', 'perf_6_month_avg', 'perf_12_month_avg', 'local_bo_qty', 'deck_risk', 'oe_constraint', 'ppap_risk', 'stop_auto_buy', 'rev_stop', 'went_on_backorder']


In [13]:
#Double check if they have the same columns

train_columns = set(df_train.columns)
test_columns = set(df_test.columns)

In [14]:
print("Columns only in training:")
print(train_columns - test_columns)

Columns only in training:
set()


In [15]:
print("Columns only in testing:")
print(test_columns - train_columns)

Columns only in testing:
set()


In [ ]:
# 0 Diferences, all ok

### Dataset information

In [17]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1687861 entries, 0 to 1687860
Data columns (total 23 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   sku                1687861 non-null  object 
 1   national_inv       1687860 non-null  float64
 2   lead_time          1586967 non-null  float64
 3   in_transit_qty     1687860 non-null  float64
 4   forecast_3_month   1687860 non-null  float64
 5   forecast_6_month   1687860 non-null  float64
 6   forecast_9_month   1687860 non-null  float64
 7   sales_1_month      1687860 non-null  float64
 8   sales_3_month      1687860 non-null  float64
 9   sales_6_month      1687860 non-null  float64
 10  sales_9_month      1687860 non-null  float64
 11  min_bank           1687860 non-null  float64
 12  potential_issue    1687860 non-null  object 
 13  pieces_past_due    1687860 non-null  float64
 14  perf_6_month_avg   1687860 non-null  float64
 15  perf_12_month_avg  1687860 non-n

In [18]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 242076 entries, 0 to 242075
Data columns (total 23 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   sku                242076 non-null  object 
 1   national_inv       242075 non-null  float64
 2   lead_time          227351 non-null  float64
 3   in_transit_qty     242075 non-null  float64
 4   forecast_3_month   242075 non-null  float64
 5   forecast_6_month   242075 non-null  float64
 6   forecast_9_month   242075 non-null  float64
 7   sales_1_month      242075 non-null  float64
 8   sales_3_month      242075 non-null  float64
 9   sales_6_month      242075 non-null  float64
 10  sales_9_month      242075 non-null  float64
 11  min_bank           242075 non-null  float64
 12  potential_issue    242075 non-null  object 
 13  pieces_past_due    242075 non-null  float64
 14  perf_6_month_avg   242075 non-null  float64
 15  perf_12_month_avg  242075 non-null  float64
 16  lo

In [19]:
# Compare data types between datasets
dtype_comparison = pd.DataFrame({
    "train_dtype": df_train.dtypes,
    "test_dtype": df_test.dtypes
})

dtype_comparison

,train_dtype,test_dtype
sku,object,object
national_inv,float64,float64
lead_time,float64,float64
in_transit_qty,float64,float64
forecast_3_month,float64,float64
forecast_6_month,float64,float64
forecast_9_month,float64,float64
sales_1_month,float64,float64
sales_3_month,float64,float64
sales_6_month,float64,float64


In [ ]:
# They are the same!

### Review if its real or synthetic data

In [21]:
# Inspect random records
df_train.sample(10, random_state=42)

,sku,national_inv,lead_time,in_transit_qty,forecast_3_month,forecast_6_month,forecast_9_month,sales_1_month,sales_3_month,sales_6_month,sales_9_month,min_bank,potential_issue,pieces_past_due,perf_6_month_avg,perf_12_month_avg,local_bo_qty,deck_risk,oe_constraint,ppap_risk,stop_auto_buy,rev_stop,went_on_backorder
732013,2079946,0.00,8.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,No,0.00,0.98,0.98,0.00,No,No,Yes,Yes,No,No
38200,1149690,16.00,8.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,2.00,0.00,No,0.00,0.70,0.75,0.00,No,No,No,Yes,No,No
1145713,1511794,2.00,2.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,No,0.00,0.92,0.96,0.00,Yes,No,Yes,Yes,No,No
1080013,1441480,3.00,8.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,0.00,No,0.00,0.62,0.62,0.00,Yes,No,No,Yes,No,No
1264726,2847275,8.00,NaN,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,No,0.00,-99.00,-99.00,0.00,Yes,No,No,No,No,No
66161,1177658,4.00,8.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,2.00,0.00,No,0.00,1.00,0.99,0.00,No,No,No,Yes,No,No
1217859,1589288,4.00,8.00,0.00,19.00,31.00,39.00,7.00,14.00,16.00,28.00,0.00,No,0.00,0.51,0.53,0.00,No,No,No,Yes,No,No
160190,1271727,3.00,8.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,No,0.00,0.99,0.98,0.00,No,No,No,Yes,No,No
390716,1738678,4.00,8.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,No,0.00,0.42,0.36,0.00,No,No,No,Yes,No,No
1564671,3159436,0.00,8.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,No,0.00,0.96,0.96,0.00,No,No,No,Yes,No,No
